In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

loans = pd.read_csv('/data/gold_loans.csv')
branch = pd.read_csv('/data/branches.csv')
repayment = pd.read_csv('/data/repayments.csv')
customers = pd.read_csv('/data/customers.csv')
gold_rates = pd.read_csv('/data/gold_rates.csv')

loans.head()


In [ ]:
#How many rows are in gold_loans exactly as received (before any cleaning)?
loans.shape

In [ ]:
# How many rows are in repayments?
repayment.shape

In [ ]:
#How many branches does Suvarna Finance have?
branch.shape

In [ ]:
# In how many different states does the company have branches?
customers['state'].value_counts()

In [ ]:
#Return one row with the earliest and latest disbursal dates as first_disbursal and last_disbursal.
answer = pd.DataFrame({ 'first_disbursal': [loans['disbursal_date'].min()],
'last_disbursal': [loans['disbursal_date'].max()]})
answer

In [ ]:
#How many distinct loan_id values are there?
loans["loan_id"].unique()

In [ ]:
#How many exact duplicate rows must be removed from gold_loans?
loans.duplicated().sum()

In [ ]:

clean = loans.drop_duplicates()
clean

In [ ]:
# In how many different ways is status written in the raw data (count distinct raw values, letter case included)?
loans['status'].nunique()

In [ ]:

loans['status'] =loans['status'].str.strip().str.lower()

In [ ]:
#After removing duplicates and converting status to lower case, return status and loans (the number of loans) for each status
answer = loans.groupby('status')['loan_id'].nunique().reset_index(name='loan_id')
answer

In [ ]:
#How many different raw values does the purity column contain?
loans['purity'].nunique()

In [ ]:
loans['karat'] = loans['purity'].apply(lambda x:
    22 if str(x).strip().upper() in ['916', '22', '22K', '22KT', '22 KT', '22.0'] else
    20 if str(x).strip().upper() in ['833', '20', '20K', '20.0'] else
    18 if str(x).strip().upper() in ['750', '18', '18K', '18.0'] else
    24 if str(x).strip().upper() in ['999', '24', '24K', '24.0'] else
    np.nan
)
loans['karat']


In [ ]:
#After removing duplicates and converting purity to a karat number (916 = 22, 833 = 20, 750 = 18, 999 = 24), return karat and loans for each kara
answer = loans.groupby('karat')['loan_id'].count().reset_index(name = 'loans')
answer

In [ ]:
#After removing duplicates, how many loans have no gold weight recorded?
loans['gold_weight_gm'].isnull().sum()

In [ ]:
#After removing duplicates, how many loans have a principal of 0 or less?
(loans['principal']<= 0).sum()

In [ ]:
#How many active loans are there after removing duplicates (status 'active' in any letter case)?
(loans['status'] == 'active').sum()

In [ ]:
#How many active loans cannot be assessed because the gold weight is missing or the principal is 0 or less?
a = clean[(loans['status'] == 'active') & (
    (loans['principal']<= 0) |
    (loans['gold_weight_gm'].isnull())
)
].shape[0]
a

In [ ]:
#How many daily rates are in gold_rates?
gold_rates['rate_date'].count()

In [ ]:
#What was the highest 24K gold rate (₹ per gram) during 2026?
gold_rates['rate_24k_per_gram'].max()

In [ ]:
gold_rates['rate_date'] = pd.to_datetime(gold_rates['rate_date'])

In [ ]:
#Return one row with peak_date: the date of the highest 24K rate in 2026.
rate_2026 = gold_rates[gold_rates['rate_date'].dt.year == 2026]
rate_2026

In [ ]:
##Return one row with peak_date: the date of the highest 24K rate in 2026.
max_rate = rate_2026['rate_24k_per_gram'].max()
answer = rate_2026[rate_2026['rate_24k_per_gram'] == max_rate][['rate_date']]
answer = answer.rename(columns={'rate_date': 'peak_date'})
answer

In [ ]:
#Which 24K rate (₹ per gram) applies on 15 October 2026?
oct_15 = gold_rates[gold_rates['rate_date'] == '2026-10-15']
rate_24k = oct_15['rate_24k_per_gram']
rate_24k

In [ ]:
# By what percentage has the 24K rate fallen from its 2026 peak to the assessment date? Round to 2 decimals.
max_rate = rate_2026['rate_24k_per_gram'].max()
assessment_rate = gold_rates[gold_rates['rate_date'] == '2026-10-15']
fallen = ((max_rate - assessment_rate) / max_date)*100
fallen

In [ ]:
#How many distinct city names appear in branches? (Old names such as Bangalore count separately.
branch['branch_name'].nunique()